<a href="https://colab.research.google.com/github/DanilaKrug/ai_miit/blob/practice-1/notebooks/01_first_llm_call.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практика 1. Первый вызов LLM и структурированный вывод

**Дисциплина:** Генеративный ИИ и ИИ-агенты для транспортной логистики (РУТ (МИИТ), магистратура)
**Время:** 90 минут. **Среда:** Google Colab.

## Что мы сегодня делаем

Клиент присылает заявку на перевозку живым текстом:

> «Нужно 15 полувагонов из Долино на Большевышнск в начале мая, груз — песок»

К концу занятия у вас будет функция, которая превращает такой текст в **валидированный
объект Pydantic** — с грузом и родом вагона из справочника, с нормализованными названиями
станций и с честно проставленным флагом неоднозначности там, где заявка неполная.

Это первый слой вашего агента: дальше в курсе на него сядут инструменты, MCP и оптимизация.

## Правило 70/30

Каркас уже написан. Вы дописываете **4 помеченные ячейки `# TODO`**:

| TODO | Функция | О чём |
|---|---|---|
| 1 | `build_naive_prompt(order_text) -> str` | наивный промпт — контрольный образец |
| 2 | `build_system_prompt(today) -> str` | инженерный system-промпт со справочниками и правилами |
| 3 | `parse_order(order_text, model) -> TransportOrder` | вызов модели + валидация ответа |
| 4 | `normalize_station(raw) -> str \| None` | приведение названия станции к справочнику |

## Критерий сдачи

Последняя ячейка ноутбука — `check()`. Она печатает построчно ✅/❌ и итог вида `10/10`.
**Работа сдана, когда `check()` даёт 10/10.** Никаких других отчётов не нужно.

## Главная мысль занятия

> **Период не додумывать.** Если в заявке написано «в начале мая» — это интервал, а не дата,
> и год в ней не указан. Модель, оставленная без правил, с удовольствием придумает и дату, и год.
> Две из десяти заявок в наборе подобраны именно так, чтобы наивный промпт на них сломался.

---
## 0. Установка зависимостей

В чистом Colab достаточно одной строки ниже. Локально — то же самое в своём venv.

In [ ]:
!pip install -q "openai>=1.40,<2" "pydantic>=2.7" "pandas>=2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 9.2 MB/s eta 0:00:00


---
## 1. Учебные данные

Нужны два файла из репозитория курса:

* `orders_raw.txt` — 10 свободных заявок так, как их пишут в почте и в мессенджере;
* `orders_expected.json` — эталонный разбор этих заявок, по нему работает `check()`.

Ячейка ниже берёт их из локального каталога, а в Colab — скачивает из репозитория курса.

In [ ]:
import json
import os
import re
from datetime import date
from pathlib import Path

import pandas as pd

# Каталог notebooks/data в публичном репозитории курса.
COURSE_DATA_URL = "https://raw.githubusercontent.com/MaximSantalov/ai_miit/main/notebooks/data"
DATA_FILES = ("orders_raw.txt", "orders_expected.json")


def _find_local_data_dir():
    candidates = [os.environ.get("COURSE_DATA_DIR"), "notebooks/data", "data", "../data", "."]
    for c in candidates:
        if not c:
            continue
        p = Path(c)
        if all((p / f).exists() for f in DATA_FILES):
            return p
    return None


def ensure_data() -> Path:
    """Возвращает каталог с учебными данными, при необходимости скачивая их."""
    local = _find_local_data_dir()
    if local is not None:
        return local

    import urllib.request

    target = Path("data")
    target.mkdir(exist_ok=True)
    for f in DATA_FILES:
        try:
            urllib.request.urlretrieve(f"{COURSE_DATA_URL}/{f}", target / f)
        except Exception as e:  # noqa: BLE001
            raise RuntimeError(
                f"Не удалось скачать {f} из репозитория курса ({type(e).__name__}: {e}).\n"
                f"Что сделать: проверьте COURSE_DATA_URL в этой ячейке или положите файлы "
                f"{list(DATA_FILES)} рядом с ноутбуком в каталог data/ вручную."
            ) from e
    return target


def load_raw_orders(path) -> dict:
    """Разбирает orders_raw.txt на словарь {номер заявки: текст заявки}."""
    orders, num, body = {}, None, []
    for block in Path(path).read_text(encoding="utf-8").split("\n---\n"):
        num, body = None, []
        for line in block.strip().splitlines():
            m = re.match(r"^#\s*Заявка\s+(\d+)\s*$", line.strip())
            if m:
                num, body = int(m.group(1)), []
                continue
            if line.strip().startswith("#"):
                continue
            body.append(line)
        if num is not None:
            orders[num] = "\n".join(body).strip()
    return orders


DATA_DIR = ensure_data()
RAW_ORDERS = load_raw_orders(DATA_DIR / "orders_raw.txt")
EXPECTED = {
    o["id"]: o
    for o in json.loads((DATA_DIR / "orders_expected.json").read_text(encoding="utf-8"))["orders"]
}

print(f"Загружено заявок: {len(RAW_ORDERS)}, эталонных разборов: {len(EXPECTED)}")
print("\n--- заявка 1 ---")
print(RAW_ORDERS[1])

Загружено заявок: 10, эталонных разборов: 10

--- заявка 1 ---
Коллеги, добрый день!
Нужно 15 полувагонов из Долино на Большевышнск в начале мая, груз - песок.
Заранее спасибо, ждём подтверждения.


---
## 2. Подключение к курсовому прокси

Мы не ходим в облако вендора напрямую. Все вызовы идут в **курсовой LiteLLM-прокси** —
он совместим с OpenAI API, поэтому пользуемся обычной библиотекой `openai`, просто с другим
`base_url`. У каждого свой ключ с ограниченным бюджетом.

На прокси два имени моделей: `course-fast` — на ней идёт всё занятие, и `course-smart` —
сильно дороже, только там, где это прямо сказано в задании. В ячейке ниже имя задаётся
один раз, переменной `MODEL`, и дальше по ноутбуку нигде не повторяется.

**Если у вас есть свой ключ** — DeepSeek или OpenAI — работайте на нём, курсовой нужен
тем, у кого своего нет. В ячейке ниже для этого закомментированы готовые пары значений:
менять надо `BASE_URL` и `MODEL` вместе, потому что имя `course-fast` существует только
на курсовом прокси.

**Где взять ключ.** На странице курса [kamani.tech/course](https://kamani.tech/course)
есть форма: вводите код группы (преподаватель называет его на занятии) и фамилию —
получаете свой ключ. Один ключ на студента, с бюджетом.

**Ключ не вставляйте в код и не коммитьте.** Ячейка ниже спросит его через `getpass`,
ввод не отобразится и в ноутбуке не сохранится. Если ошиблись при вводе — `connect(reset=True)`.

In [3]:
import getpass

from openai import OpenAI

# ── Куда ходим и какой моделью ──────────────────────────────────────────
# Вариант 1 (по умолчанию): курсовой ключ и курсовой прокси.
BASE_URL = "https://llm.kamani.tech/v1"
MODEL = "course-fast"

# Вариант 2: у вас есть свой ключ DeepSeek — раскомментируйте две строки,
# и ноутбук пойдёт напрямую к провайдеру, мимо курсового прокси.
# BASE_URL = "https://api.deepseek.com/v1"
# MODEL = "deepseek-chat"
#
# Вариант 3: свой ключ OpenAI.
# BASE_URL = "https://api.openai.com/v1"
# MODEL = "gpt-4o-mini"
#
# Больше в ноутбуке имя модели нигде не зашито: везде используется MODEL.

def connect(reset: bool = False) -> OpenAI:
    """Создаёт клиента. connect(reset=True) — ввести ключ заново.

    Ключ хранится в переменной окружения, поэтому повторный запуск ячейки его
    не переспрашивает. Если ключ скопирован с пробелом или не тот — вызовите
    connect(reset=True), и ячейка спросит его снова.
    """
    global client
    if reset:
        os.environ.pop("COURSE_LLM_API_KEY", None)
    if not os.environ.get("COURSE_LLM_BASE_URL"):
        os.environ["COURSE_LLM_BASE_URL"] = (
            BASE_URL or input("base_url (например https://.../v1): ").strip()
        )
    if not os.environ.get("COURSE_LLM_API_KEY"):
        os.environ["COURSE_LLM_API_KEY"] = getpass.getpass("Ваш ключ (ввод не виден): ").strip()
    client = OpenAI(
        base_url=os.environ["COURSE_LLM_BASE_URL"],
        api_key=os.environ["COURSE_LLM_API_KEY"],
    )
    print("Клиент создан. base_url =", os.environ["COURSE_LLM_BASE_URL"], "| модель:", MODEL)
    return client


client = connect()

Ваш ключ (ввод не виден): ··········
Клиент создан. base_url = https://llm.kamani.tech/v1 | модель: course-fast


In [4]:
def check_connection(model: str = MODEL) -> bool:
    """Проверяет связь с прокси и объясняет отказ человеческим языком."""
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Ответь одним словом: пинг"}],
            max_tokens=10,
        )
    except Exception as e:  # noqa: BLE001
        text = f"{type(e).__name__}: {e}"
        low = text.lower()
        if "401" in text or "authentication" in low or "invalid api key" in low:
            print("❌ Прокси не принял ключ (401). Проверьте, что скопировали ключ целиком "
                  "и без пробелов по краям. Чтобы ввести его заново, выполните connect(reset=True) "
                  "— простой перезапуск ячейки ключ не переспросит, он уже сохранён в окружении.")
        elif "429" in text or "budget" in low or "rate limit" in low:
            print("❌ Прокси ответил 429: исчерпан бюджет ключа или лимит запросов в минуту. "
                  "Подождите минуту; если не помогло — скажите преподавателю, бюджет поднимут.")
        elif "404" in text or "not found" in low or "does not exist" in low:
            print(f"❌ Прокси ответил 404. Две обычные причины: base_url без хвоста /v1 "
                  f"или имя модели '{model}' не подходит к этому base_url "
                  f"(курсовому прокси нужно 'course-fast', DeepSeek напрямую — 'deepseek-chat').")
        elif "connect" in low or "timeout" in low or "name or service" in low:
            print("❌ Не достучались до прокси: проверьте адрес в COURSE_LLM_BASE_URL "
                  "и что он открывается из Colab (http против https, опечатка в домене).")
        else:
            print("❌ Вызов не прошёл, причина не распознана — покажите текст ниже преподавателю.")
        print("   Технический текст ошибки:", text)
        return False

    print("✅ Связь с курсовым прокси есть. Ответ модели:", (r.choices[0].message.content or "").strip())
    return True


check_connection()

✅ Связь с курсовым прокси есть. Ответ модели: понг


True

---
## 3. Первый вызов LLM

Ниже — минимальная обёртка `ask()`, которой мы будем пользоваться весь ноутбук.
Обратите внимание на `temperature=0`: для задач извлечения данных нам не нужна
вариативность, нужна воспроизводимость.

In [5]:
def ask(user: str, system: str | None = None, model: str = MODEL, temperature: float = 0.0) -> str:
    """Один вызов чат-модели, возвращает текст ответа."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": user}
    ]
    r = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
    return (r.choices[0].message.content or "").strip()


print(ask("Одним предложением: чем полувагон (ПВ) отличается от крытого вагона (КР)?"))

Полувагон — это открытый сверху вагон с высокими бортами для сыпучих и навалочных грузов, не боящихся осадков, тогда как крытый вагон имеет жесткую крышу и закрытые стены для защиты грузов от атмосферных воздействий.


---
## 4. Справочники курса и целевая схема

Свободный текст мы приводим к **фиксированным справочникам** — тем же самым, что живут
в учебном наборе. «Песок», «гравий» и «песок с гравием» должны стать одним значением
`ПЕСОК И ГРАВИЙ`, а «полувагон» и «п/в» — родом `ПВ`, иначе следующий слой агента
(запрос к данным) не найдёт ничего. Значения справочников написаны заглавными не для
красоты: ровно так они лежат в витринах, а сверять мы будем строкой.

`TODAY` — «сегодня» курса. Мы **передаём дату в промпт явно**, а не надеемся, что модель
знает текущее число: у модели нет часов, а дата обучения к делу отношения не имеет.

В `TransportOrder` включено `extra="forbid"`: если модель допишет поле, которого нет в схеме,
валидация упадёт. Это и есть техническая формулировка запрета «не додумывать».

In [18]:
from typing import Optional

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

TODAY = date(2021, 4, 30)  # «сегодня» курса — последний день учебного набора

# Опорные станции: 30 из 300, топ по обороту учебного набора.
STATIONS = [
    "ДОЛИНО", "БОЛЬШЕВЫШНСК", "НОВОЯНТАРОДАР", "БЛИЖНЕЛИПНЫЙ", "МАЛОСЛАНЦОВО",
    "ПОДДУБОГОРСК", "КЕДРКА", "ЛЕБЯЖЬЕ", "ЗАГРАНИТАНКА", "СУХНАЯ",
    "КРУТОВО", "ОЗЕРАНКА", "ГОРКА", "БОЛЬШЕТИХЦЫ", "МЕЛОВЬЕ",
    "БЫСТРОГОРСК", "ЕЛЬНОГРАД", "ДАЛЬНЕОЛЬХОВО", "НИЖНЕМОКРНЫЙ", "НИЖНЕДОЛИНО",
    "ТИХНЫЙ", "ЖУРАВЛОВО", "НОВОГРАНИТНЫЙ", "МАЛОПОЛОГНАЯ", "МАЛОСЕРЬЕ",
    "МАЛОЗЕЛЕНАНКА", "МОКРИНО", "СУХКА", "НОВОЯБЛОНКА", "КЛЕННАЯ",
]

# Группы груза — все десять, что есть в наборе.
CARGOES = [
    "ПЕСОК И ГРАВИЙ", "ГЛИНОЗЕМ", "КАУЧУК И РЕЗИНОТЕХНИКА", "СТЕКЛОТАРА",
    "МЕБЕЛЬ И ДЕРЕВОИЗДЕЛИЯ", "ТЕКСТИЛЬ И ВОЛОКНО", "КЕРАМИКА И САНТЕХНИКА",
    "МАСЛА ТЕХНИЧЕСКИЕ", "КРАХМАЛ И ПАТОКА", "ТАРА И УПАКОВКА",
]

# Рода подвижного состава — коды, как в витринах: ПВ полувагон, КР крытый,
# ПЛ платформа, ЦС цистерна, ЦМВ цельнометаллический, ПЛФИТ фитинговая платформа.
WAGON_TYPES = ["ПВ", "КР", "ПЛ", "ЦС", "ЦМВ", "ПЛФИТ"]


class TransportOrder(BaseModel):
    """Нормализованная заявка на перевозку."""

    # extra="forbid": модель не имеет права дописать поле, которого нет в схеме
    model_config = ConfigDict(extra="forbid")

    cargo: Optional[str] = Field(None, description="груз строго из справочника CARGOES")
    wagon_type: Optional[str] = Field(None, description="род вагона строго из справочника WAGON_TYPES")
    wagons_count: Optional[int] = Field(None, ge=1, description="количество вагонов, если названо в заявке")
    from_station: Optional[str] = Field(None, description="станция отправления")
    to_station: Optional[str] = Field(None, description="станция назначения")
    date_from: Optional[date] = Field(None, description="начало периода погрузки")
    date_to: Optional[date] = Field(None, description="конец периода погрузки")
    is_ambiguous: bool = Field(False, description="в заявке есть неоднозначность, её нельзя разрешить без клиента")
    ambiguity_reason: Optional[str] = Field(None, description="что именно неоднозначно, по-русски")
    confidence: float = Field(1.0, ge=0.0, le=1.0,description="уверенность разбора от 0 до 1")

    @field_validator("cargo")
    @classmethod
    def _cargo_from_reference(cls, v):
        if v is not None and v not in CARGOES:
            raise ValueError(f"груз '{v}' отсутствует в справочнике курса: {CARGOES}")
        return v

    @field_validator("wagon_type")
    @classmethod
    def _wagon_type_from_reference(cls, v):
        if v is not None and v not in WAGON_TYPES:
            raise ValueError(f"род вагона '{v}' отсутствует в справочнике курса: {WAGON_TYPES}")
        return v

    @model_validator(mode="after")
    def _period_and_flag(self):
        if self.date_from and self.date_to and self.date_to < self.date_from:
            raise ValueError("date_to раньше date_from — период перевёрнут")
        if self.is_ambiguous and not (self.ambiguity_reason or "").strip():
            raise ValueError("поднят флаг is_ambiguous, но не заполнена ambiguity_reason")
        return self


print("Полей в схеме:", len(TransportOrder.model_fields))
print("Сегодня по условиям курса:", TODAY.isoformat())

Полей в схеме: 10
Сегодня по условиям курса: 2021-04-30


---
## 5. Как этот ноутбук сообщает о невыполненных заданиях

В ячейках с заданием стоит пустое место с подписью «здесь ваш ответ». Пока оно пустое,
функция ничего не возвращает — и готовый код это замечает сам: вместо трассировки на
пол-экрана вы увидите строку о том, какое задание ещё не сделано.

In [20]:
class TodoNotDone(NotImplementedError):
    """Помеченная ячейка # TODO ещё не дописана: функция ничего не вернула."""


def need(value, number: int, what: str):
    """Проверяет результат вашей функции. Готовый код, менять не нужно."""
    if value is None or (isinstance(value, str) and not value.strip()):
        raise TodoNotDone(f"TODO {number} не выполнен: {what}")
    return value


def explain_todo(e: TodoNotDone) -> None:
    """Печатает понятное сообщение вместо трассировки."""
    print("⛔", e)
    print("   Что делать: допишите ячейку с заданием выше, выполните её (Shift+Enter)")
    print("   и запустите эту ячейку заново.")

---
## 6. TODO 1 — наивный промпт

Первый промпт пишем так, как его пишет любой человек в первый раз: одна фраза, никаких
правил. Это **контрольный образец**: в разделе 11 мы сравним его с инженерным промптом
на одной и той же заявке и увидим, где именно он ломается.

В ячейке два пустых места: сама функция и вызов модели под ней. Заполните оба и
запустите ячейку — она напечатает ответ. Прочитайте его глазами, прежде чем идти
дальше, и задайте ему три вопроса:

* **Дата.** Какое там число и какой год — и откуда модель его взяла, если в заявке года нет?
* **Станции.** Пришли как в справочнике или как их написал клиент?
* **Неопределённость.** Есть ли в ответе хоть какой-то способ сказать «я не знаю»?

Ответ, скорее всего, будет выглядеть прилично. В этом и дело: наивный промпт ломается
не шумом, а правдоподобной выдумкой.

In [21]:
# ═══════════════════════════ TODO 1 из 4 ═══════════════════════════
# Задача из двух частей: собрать наивный промпт функцией ниже и вызвать
# им модель на заявке 1, чтобы увидеть, что она вернёт.
#
# Наивный промпт — это одна-две фразы: никаких справочников, никакой текущей
# даты, никаких правил. Попросите вернуть JSON с полями: груз, род вагона,
# количество вагонов, станция отправления, станция назначения, дата — одна,
# именно так, наивно. Текст заявки order_text подставьте в промпт целиком,
# как он есть, ничего в нём не меняя.
# ═══════════════════════════════════════════════════════════════════


def build_naive_prompt(order_text: str) -> str:
    return f"""Разбери заявку на перевозку и верни JSON с полями:
груз, род вагона, количество вагонов, станция отправления, станция назначения, дата.

Заявка:
{order_text}"""



demo_text = RAW_ORDERS[1]
naive_raw = ask(build_naive_prompt(demo_text), model=MODEL)

# ──────────────────────────────────────────────────────────────────
# здесь вызовите модель с промптом и моделью, ответ положите в naive_raw:
#     ask(build_naive_prompt(demo_text), model=MODEL)
# ──────────────────────────────────────────────────────────────────


# ── Ниже менять не нужно: печать результата ──
if not str(naive_raw).strip():
    print("⛔ Ответа нет. Проверьте две вещи: написан ли промпт в функции выше "
          "и присвоен ли результат вызова переменной naive_raw.")
else:
    print("─── заявка ───")
    print(demo_text)
    print("\n─── что вернула модель на ваш наивный промпт ───")
    print(naive_raw)
    print("\nПосмотрите на поле даты: что в нём стоит и откуда это взялось?")

─── заявка ───
Коллеги, добрый день!
Нужно 15 полувагонов из Долино на Большевышнск в начале мая, груз - песок.
Заранее спасибо, ждём подтверждения.

─── что вернула модель на ваш наивный промпт ───
```json
{
  "груз": "песок",
  "род_вагона": "полувагон",
  "количество_вагонов": 15,
  "станция_отправления": "Долино",
  "станция_назначения": "Большевышнск",
  "дата": "начало мая"
}
```

Посмотрите на поле даты: что в нём стоит и откуда это взялось?


---
## 7. TODO 2 — инженерный system-промпт

Теперь то же самое, но по-инженерному. Инженерный промпт отличается от наивного не длиной,
а тем, что в нём есть **контекст, словарь и правило поведения в неопределённости**.

In [22]:
# ═══════════════════════════ TODO 2 из 4 ═══════════════════════════
# Что здесь написать: system-промпт для разбора заявок.
#
# Сигнатура:  build_system_prompt(today: date) -> str
#
# Обязательно должно быть в тексте промпта (это проверяет check()):
#   1) текущая дата в формате ISO — подставьте today.isoformat(), а не литерал:
#      модель не знает, какое сегодня число, ей нужно это сказать;
#   2) справочники: перечислите станции (STATIONS), грузы (CARGOES) и рода вагонов
#      (WAGON_TYPES) и потребуйте выбирать значения строго из них;
#   3) имена полей ответа: cargo, wagon_type, wagons_count, from_station, to_station,
#      date_from, date_to, is_ambiguous, ambiguity_reason — и запрет добавлять свои поля;
#   4) правило «период не додумывать» — сформулируйте своими словами, но со словом
#      «додумывать» (или «придумывать»), и раскройте его:
#        • нечёткий период («в начале мая») -> границы диапазона в date_from/date_to
#          плюс is_ambiguous = true и объяснение в ambiguity_reason;
#        • год не указан -> берём год из переданной текущей даты, а не из своих знаний;
#        • поле не названо в заявке -> null, догадка запрещена
#          (объём в тоннах НЕ пересчитывается в вагоны);
#   5) требование вернуть один JSON-объект без markdown-обрамления, причина — по-русски.
# ═══════════════════════════════════════════════════════════════════


def build_system_prompt(today: date) -> str:
    return f"""Ты разбираешь заявки на перевозку грузов в вагонах и превращаешь
свободный текст в строгий JSON-объект.

Сегодняшняя дата: {today.isoformat()}.

Справочники — значения нужно выбирать СТРОГО из них, никаких других вариантов:
- станции: {", ".join(STATIONS)}
- грузы: {", ".join(CARGOES)}
- рода вагонов: {", ".join(WAGON_TYPES)}

Ответ должен быть ровно одним JSON-объектом со следующими полями и никакими другими:
cargo, wagon_type, wagons_count, from_station, to_station, date_from, date_to,
is_ambiguous, ambiguity_reason.
Не добавляй никаких полей, кроме перечисленных.

Главное правило: период погрузки нельзя додумывать или придумывать. Раскрывается
это так:
- если период в заявке нечёткий (например, «в начале мая», «в первой половине месяца»),
  переведи его в границы диапазона date_from и date_to, поставь is_ambiguous = true
  и по-русски объясни причину в ambiguity_reason;
- если в заявке не указан год, возьми год из переданной сегодняшней даты
  ({today.isoformat()}), а не из своих знаний о мире;
- если какое-то поле в заявке вообще не названо, ставь null и не догадывайся —
  например, объём груза в тоннах никогда не пересчитывается в количество вагонов.

Верни только сам JSON-объект, без пояснений и без markdown-обрамления
(без ```json и без ```). Текст в ambiguity_reason пиши по-русски.

Дополнительно верни поле confidence — число от 0 до 1, насколько ты уверен в разборе.
Уменьшай confidence, если: какие-то поля пришлось поставить null (не названы в заявке),
если is_ambiguous = true (период указан нечётко), или если станцию не удалось
однозначно сопоставить со справочником. Если все поля определены точно и period
указан явно — confidence должен быть высоким (0.9–1.0)."""


# ── Ниже менять не нужно: сравнение с наивным промптом ────────────
demo_text = RAW_ORDERS[1]
system_raw = ""
try:
    system_prompt = need(build_system_prompt(TODAY), 2,
                         "функция build_system_prompt ничего не вернула")
    system_raw = ask(demo_text, system=system_prompt, model=MODEL)

    print("─── заявка ───")
    print(demo_text)
    print("\n─── что вернула модель по вашему инженерному промпту ───")
    print(system_raw)
    print("\nСравните с ответом из TODO 1: что стало с датой и откуда взялся год?")
except TodoNotDone as e:
    explain_todo(e)

─── заявка ───
Коллеги, добрый день!
Нужно 15 полувагонов из Долино на Большевышнск в начале мая, груз - песок.
Заранее спасибо, ждём подтверждения.

─── что вернула модель по вашему инженерному промпту ───
{
  "cargo": "ПЕСОК И ГРАВИЙ",
  "wagon_type": "ПВ",
  "wagons_count": 15,
  "from_station": "ДОЛИНО",
  "to_station": "БОЛЬШЕВЫШНСК",
  "date_from": "2021-05-01",
  "date_to": "2021-05-10",
  "is_ambiguous": true,
  "ambiguity_reason": "Период погрузки указан нечётко: «в начале мая» — границы диапазона определены как 1–10 мая.",
  "confidence": 0.75
}

Сравните с ответом из TODO 1: что стало с датой и откуда взялся год?


---
## 8. Структурированный вывод: как просить JSON

Просить JSON словами недостаточно — модель периодически обернёт его в ```` ```json ````.
Поэтому мы (а) просим режим `json_object`, (б) всё равно чистим ответ,
(в) **всё равно** валидируем его Pydantic.

Три рубежа вместо одного — потому что каждый из них иногда протекает.

Важная деталь про режимы. `json_object` гарантирует только одно: ответ будет
синтаксически корректным JSON. Он **не** гарантирует, что в нём будут наши поля.
Есть более строгий режим `json_schema`, где провайдеру отдаётся сама схема, но
умеют его не все — DeepSeek, на котором идёт курс, не умеет. Поэтому договор о
полях мы держим на своей стороне: перечисляем их в промпте и проверяем `Pydantic`.
Это не обходной путь, а нормальная инженерная позиция: чужую гарантию, которой
может не быть, заменяем своей проверкой, которая есть всегда.

Переменная `JSON_SCHEMA` ниже — та самая схема, сгенерированная из модели
Pydantic. В TODO 2 она пригодится: перечень полей в промпт удобно брать из неё,
а не переписывать руками.

Эта ячейка готова, менять её не нужно.

In [23]:
JSON_SCHEMA = TransportOrder.model_json_schema()


def call_json(system: str, user: str, model: str = MODEL) -> dict:
    """Вызывает модель и возвращает распарсенный JSON (ещё не валидированный).

    response_format="json_object" — это всё, на что можно рассчитывать у любого
    провайдера: гарантируется синтаксически валидный JSON, но не наши поля.
    Строгую схему (json_schema) умеют не все, DeepSeek — не умеет. Поэтому
    договор о полях держим на своей стороне: схему кладём в промпт, а ответ
    проверяем Pydantic'ом. Это и есть урок занятия.
    """
    need(system, 2, "system-промпт пустой — функция build_system_prompt ничего не вернула")
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    r = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"},
    )

    raw = (r.choices[0].message.content or "").strip()
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"модель вернула не JSON: {raw[:300]!r}") from e

---
## 9. TODO 3 — разбор одной заявки

Собираем всё вместе: инженерный промпт + вызов + валидация.

In [24]:
# ═══════════════════════════ TODO 3 из 4 ═══════════════════════════
# Что здесь написать: разбор одной заявки в объект TransportOrder.
#
# Сигнатура:  parse_order(order_text: str, model: str = MODEL) -> TransportOrder
#
# Три шага, все части уже есть выше:
#   1) собрать system-промпт: build_system_prompt(TODAY);
#   2) получить словарь: call_json(system, user, model=model),
#      где user — текст заявки (можно с короткой подводкой «Заявка:\n...»);
#   3) вернуть TransportOrder.model_validate(данные).
#
# Функция обязана вернуть именно объект TransportOrder, а не словарь:
# check() проверяет тип. Исключения валидации не глушите — их ловит ячейка прогона.
# ═══════════════════════════════════════════════════════════════════


def parse_order(order_text: str, model: str = MODEL) -> TransportOrder:
    system = build_system_prompt(TODAY)
    data = call_json(system, f"Заявка:\n{order_text}", model=model)
    return TransportOrder.model_validate(data)

---
## 10. TODO 4 — нормализация станций

Справочник в промпте помогает, но не гарантирует. Клиент пишет «Долино», «ст. Кедрка»,
«из Большевышнска» — и модель нередко возвращает это как есть, в своём регистре и падеже.
Второй рубеж делаем
**детерминированным**: обычная функция на Python, без LLM, которую можно протестировать.

Это ровно тот приём, который в практике 4 превратится в semantic layer:
модель предлагает — код проверяет и приводит к канону.

In [25]:
# ═══════════════════════════ TODO 4 из 4 ═══════════════════════════
# Что здесь написать: приведение названия станции к справочнику STATIONS.
#
# Сигнатура:  normalize_station(raw: Optional[str]) -> Optional[str]
#
# Помощники ниже уже готовы, менять их не нужно. Ваша задача — написать только
# normalize_station(), применив правило из пунктов 1–6.
#
# Правила:
#   1) None или пустая строка -> None;
#   2) отбросить служебные префиксы: "ст.", "ст ", "станция";
#   3) регистр и буква ё не важны: "долино" == "ДОЛИНО". Приводит к общему виду
#      готовый помощник _canon() — и пропускать через него надо ОБЕ сравниваемые
#      строки, и запрос, и справочное имя. Справочник STATIONS написан заглавными,
#      а _canon() переводит в строчные: сравнение "ДОЛИНО" с _canon("Долино") даёт
#      ноль общих букв, и функция вернёт None вообще на всех случаях;
#   4) точное совпадение со справочником -> вернуть справочное имя
#      (сравниваем _canon(станция) == _canon(запрос), а возвращаем станцию как есть);
#   5) иначе разбираемся с падежами по общему началу слова: посчитайте для каждой
#      станции длину общего префикса с запросом — снова по _canon с обеих сторон —
#      и возьмите станцию с самым длинным.
#      Засчитывать только при трёх условиях сразу: префикс не короче 6 символов,
#      строго длиннее, чем у второй по близости станции, И покрывает не меньше 80%
#      длины справочного имени:
#      "Большевышнска" -> "БОЛЬШЕВЫШНСК", "Новогранитного" -> "НОВОГРАНИТНЫЙ",
#      а вот "Дальнеборск" -> None: общее начало с "ДАЛЬНЕОЛЬХОВО" всего 6 букв
#      из 13, это приставка ДАЛЬНЕ-, а не падеж;
#   6) если ни одна не дотянула или лидеров несколько -> None.
#      None здесь означает «сопоставить не удалось» — это честнее, чем угадать.
#
# check() проверит на: "Долино", "долино", "ст. Кедрка", "станция Горка",
# "Большевышнска", "Новогранитного", а также на "Дальнеборск" и на заведомо
# чужом "Заречное-Товарное" — на обоих ожидается None.
# ═══════════════════════════════════════════════════════════════════

_PREFIXES = ("станция ", "ст. ", "ст.", "ст ")


def _canon(s: str) -> str:
    """Готовый помощник: убирает служебные различия в названии."""
    s = s.strip().lower().replace("ё", "е")
    for prefix in _PREFIXES:
        if s.startswith(prefix):
            s = s[len(prefix):].strip()
            break
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    return " ".join(s.split()).strip(" -")


MIN_STEM = 6      # короче — это уже не падеж, а другая станция
MIN_COVER = 0.8   # общее начало должно покрыть 80% справочного имени


def _common_prefix(a: str, b: str) -> int:
    """Готовый помощник: считает совпадающие символы с начала строк."""
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


def normalize_station(raw: Optional[str]) -> Optional[str]:
    if not raw or not raw.strip():
        return None

    query = _canon(raw)
    if not query:
        return None

    # 4) точное совпадение
    for station in STATIONS:
        if _canon(station) == query:
            return station

    # 5) иначе — станция с самым длинным общим началом (падежи)
    best_station = None
    best_len = 0
    second_len = 0
    for station in STATIONS:
        common = _common_prefix(_canon(station), query)
        if common > best_len:
            second_len = best_len
            best_len = common
            best_station = station
        elif common > second_len:
            second_len = common

    if best_station is None:
        return None

    cover = best_len >= MIN_COVER * len(_canon(best_station))
    if best_len >= MIN_STEM and best_len > second_len and cover:
        return best_station

    # 6) ни одна не дотянула или лидеров несколько
    return None


def parse_order_normalized(order_text: str, model: str = MODEL) -> TransportOrder:
    """Разбор заявки + нормализация станций. Готовый код, менять не нужно.

    Пока TODO 4 не сделан, normalize_station возвращает None, и станция
    остаётся в том виде, в каком её вернула модель, — прогон на этом не встаёт.
    """
    order = need(parse_order(order_text, model=model), 3,
                 "функция parse_order ничего не вернула")
    order.from_station = normalize_station(order.from_station) or order.from_station
    order.to_station = normalize_station(order.to_station) or order.to_station
    return order

### Быстрая проверка нормализации

Эта ячейка готова. Выполните её после TODO 4: она показывает, какие названия удалось
привести к справочнику и где функция должна честно вернуть `None`, а не угадать станцию.

In [26]:
station_examples = [
    ("Долино", "ДОЛИНО"),
    ("ст. Кедрка", "КЕДРКА"),
    ("Большевышнска", "БОЛЬШЕВЫШНСК"),
    ("Новогранитного", "НОВОГРАНИТНЫЙ"),
    ("Дальнеборск", None),
    ("Заречное-Товарное", None),
]

rows = []
for raw, expected in station_examples:
    got = normalize_station(raw)
    rows.append({
        "клиент написал": raw,
        "результат": got,
        "ожидалось": expected,
        "верно": "✅" if got == expected else "❌",
    })
pd.DataFrame(rows)

,клиент написал,результат,ожидалось,верно
0,Долино,ДОЛИНО,ДОЛИНО,✅
1,ст. Кедрка,КЕДРКА,КЕДРКА,✅
2,Большевышнска,БОЛЬШЕВЫШНСК,БОЛЬШЕВЫШНСК,✅
3,Новогранитного,НОВОГРАНИТНЫЙ,НОВОГРАНИТНЫЙ,✅
4,Дальнеборск,None,None,✅
5,Заречное-Товарное,None,None,✅


---
## 11. Наивный против инженерного: одна заявка, два результата

Та же заявка 1, «в начале мая». Ответ наивного промпта у нас уже есть — он остался в
переменной `naive_raw` из раздела 6. Теперь разберём ту же заявку инженерным путём и
положим два результата рядом.

Ничего дописывать здесь не нужно: выполните две готовые ячейки ниже и прочитайте таблицу.
Она нужна, чтобы рядом увидеть результат наивного промпта из TODO 1 и результат после
всех четырёх TODO. Если ячейка из раздела 6 не была выполнена, сравнения не получится.

In [27]:
engineered = None
try:
    engineered = parse_order_normalized(demo_text)
    print("─── ответ на ИНЖЕНЕРНЫЙ промпт (после валидации) ───")
    print(engineered.model_dump_json(indent=2))
except TodoNotDone as e:
    explain_todo(e)

─── ответ на ИНЖЕНЕРНЫЙ промпт (после валидации) ───
{
  "cargo": "ПЕСОК И ГРАВИЙ",
  "wagon_type": "ПВ",
  "wagons_count": 15,
  "from_station": "ДОЛИНО",
  "to_station": "БОЛЬШЕВЫШНСК",
  "date_from": "2021-05-01",
  "date_to": "2021-05-10",
  "is_ambiguous": true,
  "ambiguity_reason": "Период погрузки указан нечётко: «в начале мая», границы диапазона определены как 1–10 мая.",
  "confidence": 0.75
}


In [28]:
def naive_to_dict(text: str) -> dict:
    """Достаёт первый JSON-объект из свободного ответа модели."""
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}


def pick(d: dict, *aliases) -> str:
    """Ищет значение по любому из имён поля — наивный промпт схему не фиксирует."""
    norm = {str(k).strip().lower().replace(" ", "_"): v for k, v in d.items()}
    for a in aliases:
        if a in norm and norm[a] not in (None, ""):
            return str(norm[a])
    return "— (поля нет)"


naive = naive_to_dict(naive_raw)
comparison = None

if engineered is None:
    print("Таблица сравнения появится, когда будут дописаны TODO 1-4 "
          "и выполнены две ячейки выше.")
    rows = []
else:
    rows = [
        ["Период погрузки",
         pick(naive, "дата", "date", "дата_погрузки"),
         f"{engineered.date_from} … {engineered.date_to}"],
        ["Откуда взят год",
         "из весов модели — мы его не задавали",
         f"из переданной текущей даты {TODAY.isoformat()}"],
        ["Станция отправления",
         pick(naive, "станция_отправления", "from_station", "откуда"),
         str(engineered.from_station)],
        ["Станция назначения",
         pick(naive, "станция_назначения", "to_station", "куда"),
         str(engineered.to_station)],
        ["Груз",
         pick(naive, "груз", "cargo"),
         str(engineered.cargo)],
        ["Род вагона",
         pick(naive, "род_вагона", "wagon_type", "тип_вагона"),
         str(engineered.wagon_type)],
        ["Признак неоднозначности",
         "в схеме отсутствует",
         f"{engineered.is_ambiguous} — {engineered.ambiguity_reason}"],
        ["Проверка значений",
         "никакой: сойдёт любая строка",
         "Pydantic + справочники курса + нормализация станций"],
    ]
    pd.set_option("display.max_colwidth", 60)
    comparison = pd.DataFrame(
        rows, columns=["Что сравниваем", "Наивный промпт", "Инженерный промпт"]
    )

comparison

,Что сравниваем,Наивный промпт,Инженерный промпт
0,Период погрузки,начало мая,2021-05-01 … 2021-05-10
1,Откуда взят год,из весов модели — мы его не задавали,из переданной текущей даты 2021-04-30
2,Станция отправления,Долино,ДОЛИНО
3,Станция назначения,Большевышнск,БОЛЬШЕВЫШНСК
4,Груз,песок,ПЕСОК И ГРАВИЙ
5,Род вагона,полувагон,ПВ
6,Признак неоднозначности,в схеме отсутствует,"True — Период погрузки указан нечётко: «в начале мая», г..."
7,Проверка значений,никакой: сойдёт любая строка,Pydantic + справочники курса + нормализация станций


Разберите таблицу глазами, прежде чем идти дальше. Три различия, ради которых всё затевалось:

1. **период** — у наивного одна дата, у инженерного диапазон;
2. **год** — у наивного он взялся неизвестно откуда, у инженерного из переданного «сегодня»;
3. **сигнал о неполноте** — наивный молча уверен, инженерный говорит, чего он не знает.

Третье — самое важное для агента. Агент, который не умеет сказать «не знаю»,
не умеет и переспросить.

---
## 12. Прогон по всем 10 заявкам

10 вызовов модели — примерно полминуты. Ошибки не прерывают цикл: неразобранные заявки
попадут в `ERRORS` с текстом причины.

Это готовая ячейка: она запускает `parse_order_normalized` для всех 10 заявок, сохраняет
удачные разборы в `RESULTS`, а ошибки — в `ERRORS`. Ничего писать не нужно: выполните её
перед `check()`, иначе проверке будет не с чем сравнивать результат.

In [29]:
RESULTS: dict[int, TransportOrder] = {}
ERRORS: dict[int, str] = {}

for i in sorted(RAW_ORDERS):
    try:
        RESULTS[i] = parse_order_normalized(RAW_ORDERS[i])
        print(f"заявка {i:>2}: ок")
    except TodoNotDone as e:
        print(f"заявка {i:>2}: {e}")
        print("Прогон остановлен: сначала допишите помеченную ячейку и выполните её заново.")
        break
    except Exception as e:  # noqa: BLE001
        ERRORS[i] = f"{type(e).__name__}: {e}"
        print(f"заявка {i:>2}: ошибка — {ERRORS[i][:200]}")

print(f"\nРазобрано: {len(RESULTS)} из {len(RAW_ORDERS)}")

заявка  1: ок
заявка  2: ок
заявка  3: ок
заявка  4: ок
заявка  5: ок
заявка  6: ок
заявка  7: ок
заявка  8: ок
заявка  9: ок
заявка 10: ок

Разобрано: 10 из 10


In [32]:
table = pd.DataFrame(
    [
        {
            "№": i,
            "груз": o.cargo,
            "род вагона": o.wagon_type,
            "ваг.": o.wagons_count,
            "откуда": o.from_station,
            "куда": o.to_station,
            "с": o.date_from,
            "по": o.date_to,
            "?": "да" if o.is_ambiguous else "",
            "причина": (o.ambiguity_reason or "")[:45],
            "conf": o.confidence,
        }
        for i, o in sorted(RESULTS.items())
    ]
)
if table.empty:
    print("Разобранных заявок пока нет: допишите TODO 1-4 и выполните ячейку прогона выше.")
table

,№,груз,род вагона,ваг.,откуда,куда,с,по,?,причина,conf
0,1,ПЕСОК И ГРАВИЙ,ПВ,15.0,ДОЛИНО,БОЛЬШЕВЫШНСК,2021-05-01,2021-05-10,да,Период погрузки указан нечётко: «в начале мая,0.75
1,2,СТЕКЛОТАРА,КР,30.0,НОВОЯНТАРОДАР,ПОДДУБОГОРСК,2021-05-05,2021-05-05,,,1.00
2,3,МАСЛА ТЕХНИЧЕСКИЕ,ЦС,12.0,БЛИЖНЕЛИПНЫЙ,СУХНАЯ,2021-05-07,2021-05-07,,,0.95
3,4,ГЛИНОЗЕМ,ПВ,NaN,НОВОГРАНИТНЫЙ,КЕДРКА,2021-05-11,2021-05-11,,,0.90
4,5,КЕРАМИКА И САНТЕХНИКА,None,20.0,ЛЕБЯЖЬЕ,ЗАГРАНИТАНКА,2021-05-14,2021-05-14,,,0.85
5,6,ПЕСОК И ГРАВИЙ,ПВ,40.0,КРУТОВО,ОЗЕРАНКА,2021-05-12,2021-05-18,,,0.95
6,7,ТЕКСТИЛЬ И ВОЛОКНО,ЦМВ,25.0,ГОРКА,БОЛЬШЕТИХЦЫ,2021-05-16,2021-05-16,,,0.95
7,8,ТАРА И УПАКОВКА,ПЛФИТ,60.0,МЕЛОВЬЕ,БЫСТРОГОРСК,2021-05-19,2021-05-19,,,0.95
8,9,МЕБЕЛЬ И ДЕРЕВОИЗДЕЛИЯ,ПЛ,35.0,ЕЛЬНОГРАД,ДАЛЬНЕОЛЬХОВО,2021-05-21,2021-05-21,,,0.95
9,10,КАУЧУК И РЕЗИНОТЕХНИКА,КР,18.0,БОЛЬШЕВЫШНСК,НИЖНЕДОЛИНО,2021-05-24,2021-05-24,,,1.00


---
## 13. Домашнее задание

1. **Довести `check()` до 10/10.** Если какая-то заявка упорно не сходится — правьте промпт,
   а не эталон и не проверки. Эталон и `check()` менять нельзя.

2. **Добавить в `TransportOrder` поле `confidence`** — уверенность разбора, `float` от 0 до 1:

   * поле объявляется в схеме (`extra="forbid"` этому не мешает — запрещены только
     **необъявленные** поля), описывается в `build_system_prompt` и заполняется моделью;
   * в промпте должно быть сказано, **от чего** уверенность зависит: сколько полей осталось
     `null`, поднят ли флаг неоднозначности, пришлось ли нормализовать станции;
   * `check()` после этого обязан по-прежнему давать 10/10.

3. **Короткое обоснование в markdown-ячейке в конце ноутбука** (5–7 предложений):
   почему выбрана именно такая шкала, какие значения `confidence` получились у заявок 1, 4, 5
   (три неоднозначные) против 2, 3, 7 (три чистые), и что вы будете делать со значением ниже
   порога, когда этот разбор станет входом агента.

**Критерий приёмки:** `check()` = 10/10 при объявленном поле `confidence`;
у трёх неоднозначных заявок `confidence` строго ниже, чем у трёх чистых;
обоснование на месте. Сдача — PR в свой форк репозитория курса.

---
## 14. `check()` — критерий сдачи

Десять проверок. Работа сдана при **10/10**.
Проверки устойчивы к вариативности модели: сверяются нормализованные поля,
для нечёткого периода задан допуск в днях, свободный текст причины посимвольно не сличается.

Это готовая ячейка: она сверяет `RESULTS` с эталоном и печатает результат каждой из десяти
проверок. Выполняйте её после прогона, а после любой правки промпта — сначала заново прогон,
затем проверку.

In [31]:
def _fmt(v):
    return "null" if v is None else f"'{v}'"


def _date_ok(got, expected_iso: str, tolerance_days: int) -> bool:
    if got is None:
        return False
    exp = date.fromisoformat(expected_iso)
    return abs((got - exp).days) <= tolerance_days


def _need_results():
    if not RESULTS:
        raise AssertionError(
            "нет результатов прогона: выполните ячейку «Прогон по всем 10 заявкам» выше"
        )


# ── сами проверки: каждая возвращает (успех, сообщение) ─────────────────────

def _check_schema():
    need = {"cargo", "wagon_type", "wagons_count", "from_station", "to_station",
            "date_from", "date_to", "is_ambiguous", "ambiguity_reason"}
    have = set(TransportOrder.model_fields)
    missing = sorted(need - have)
    if missing:
        return False, f"в TransportOrder нет обязательных полей: {missing}"
    if TransportOrder.model_config.get("extra") != "forbid":
        return False, ("в TransportOrder не выставлен extra='forbid' — модель сможет "
                       "дописать поле, которого нет в схеме")
    return True, f"поля на месте, лишние запрещены (extra='forbid'), объявлено {len(have)}"


def _check_todo1():
    p = build_naive_prompt(RAW_ORDERS[1])
    if p is None:
        return False, "build_naive_prompt ничего не вернула — TODO 1 не выполнен"
    if not isinstance(p, str):
        return False, f"build_naive_prompt вернула {type(p).__name__}, а нужна строка"
    if len(p.strip()) < 20:
        return False, "наивный промпт пустой или слишком короткий"
    if RAW_ORDERS[1].strip() not in p:
        return False, "текст заявки не подставлен в промпт целиком"
    if "json" not in p.lower():
        return False, "в промпте не сказано, что ответ нужен в JSON"

    # Промпт мало написать — его надо выполнить и увидеть ответ. Ответ живёт
    # в naive_raw из раздела 6; он же потом идёт в таблицу сравнения.
    raw = (globals().get("naive_raw") or "").strip()
    if not raw:
        return False, (f"промпт собирается ({len(p)} символов), но ни разу не выполнен — "
                       "запустите ячейку под TODO 1 и прочитайте ответ модели")

    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return True, (f"промпт собран ({len(p)} символов) и выполнен; модель вернула не JSON, "
                      "а свободный текст — это тоже результат наивного промпта")
    try:
        got = json.loads(m.group(0))
    except json.JSONDecodeError:
        return True, (f"промпт собран ({len(p)} символов) и выполнен; в ответе есть фигурные "
                      "скобки, но это не валидный JSON — наивный промпт формат не гарантирует")

    date_key = next((k for k in got if "дат" in str(k).lower() or "date" in str(k).lower()), None)
    head = f"промпт собран ({len(p)} символов) и выполнен"
    if date_key is None:
        return True, f"{head}; в ответе {len(got)} полей, поля даты среди них нет"

    value = got[date_key]
    year = re.search(r"\b(19|20)\d{2}\b", str(value))
    if year:
        verdict = f"год {year.group(0)} модель подставила сама, в заявке его нет"
    else:
        verdict = ("года нет и здесь: период остался словами, одно поле «дата» "
                   "диапазон не вмещает")
    return True, f"{head}; модель вернула {date_key!r} = {value!r} — {verdict}"


def _check_todo2():
    s = build_system_prompt(TODAY)
    if s is None:
        return False, "build_system_prompt ничего не вернула — TODO 2 не выполнен"
    if not isinstance(s, str):
        return False, f"build_system_prompt вернула {type(s).__name__}, а нужна строка"
    problems = []
    if TODAY.isoformat() not in s:
        problems.append(f"нет текущей даты {TODAY.isoformat()} — модель возьмёт год из своих знаний")
    if not any(k in s.lower() for k in ("додум", "придум", "угад")):
        problems.append("нет правила «период не додумывать»")
    for f in ("date_from", "date_to", "is_ambiguous"):
        if f not in s:
            problems.append(f"не названо поле ответа {f}")
    if sum(st in s for st in STATIONS) < 5:
        problems.append("в промпте нет справочника станций")
    if sum(c in s for c in CARGOES) < 5:
        problems.append("в промпте нет справочника грузов")
    if sum(w in s for w in WAGON_TYPES) < 5:
        problems.append("в промпте нет справочника родов вагонов")
    if problems:
        return False, "; ".join(problems)
    return True, f"промпт содержит дату, три справочника и правило, длина {len(s)} символов"


def _check_todo4():
    cases = [
        ("Долино", "ДОЛИНО"),
        ("долино", "ДОЛИНО"),
        ("ст. Кедрка", "КЕДРКА"),
        ("станция Горка", "ГОРКА"),
        ("Большевышнска", "БОЛЬШЕВЫШНСК"),
        ("Новогранитного", "НОВОГРАНИТНЫЙ"),
        ("Дальнеборск", None),
        ("Заречное-Товарное", None),
        ("", None),
    ]
    bad = []
    for raw, want in cases:
        got = normalize_station(raw)
        if got != want:
            bad.append(f"normalize_station({raw!r}) -> {_fmt(got)}, ожидалось {_fmt(want)}")
    if bad:
        return False, "; ".join(bad)
    return True, f"{len(cases)} случаев нормализации разобраны верно"


def _check_all_parsed():
    _need_results()
    if ERRORS:
        details = "; ".join(f"заявка {i}: {t[:90]}" for i, t in sorted(ERRORS.items()))
        return False, f"не разобрались {len(ERRORS)} заявок — {details}"
    missing = sorted(set(EXPECTED) - set(RESULTS))
    if missing:
        return False, f"нет результата по заявкам {missing}"
    wrong_type = [i for i, o in RESULTS.items() if not isinstance(o, TransportOrder)]
    if wrong_type:
        return False, f"parse_order вернул не TransportOrder для заявок {sorted(wrong_type)}"
    return True, f"все {len(RESULTS)} заявок разобраны и прошли валидацию Pydantic"


def _check_stations():
    _need_results()
    bad = []
    for i, exp in sorted(EXPECTED.items()):
        got = RESULTS.get(i)
        if got is None:
            bad.append(f"заявка {i}: нет результата")
            continue
        for field in ("from_station", "to_station"):
            v = getattr(got, field)
            if v not in STATIONS:
                bad.append(f"заявка {i}, {field}: {_fmt(v)} нет в справочнике станций")
            elif v != exp[field]:
                bad.append(f"заявка {i}, {field}: {_fmt(v)}, ожидалось {_fmt(exp[field])}")
    if bad:
        return False, "; ".join(bad[:4]) + (f" (и ещё {len(bad) - 4})" if len(bad) > 4 else "")
    return True, "все 20 станций нормализованы к справочнику и совпали с эталоном"


def _check_cargo_and_wagon():
    _need_results()
    bad = []
    for i, exp in sorted(EXPECTED.items()):
        got = RESULTS.get(i)
        if got is None:
            bad.append(f"заявка {i}: нет результата")
            continue
        if got.cargo != exp["cargo"]:
            bad.append(f"заявка {i}, груз: {_fmt(got.cargo)}, ожидалось {_fmt(exp['cargo'])}")
        if got.wagon_type != exp["wagon_type"]:
            hint = " (род вагона в заявке не назван — его нельзя выводить из груза)" \
                if exp["wagon_type"] is None else ""
            bad.append(
                f"заявка {i}, род вагона: {_fmt(got.wagon_type)}, "
                f"ожидалось {_fmt(exp['wagon_type'])}{hint}"
            )
    if bad:
        return False, "; ".join(bad[:4]) + (f" (и ещё {len(bad) - 4})" if len(bad) > 4 else "")
    return True, "груз и род вагона совпали с эталоном во всех 10 заявках"


def _check_counts():
    _need_results()
    bad = []
    for i, exp in sorted(EXPECTED.items()):
        got = RESULTS.get(i)
        if got is None:
            bad.append(f"заявка {i}: нет результата")
            continue
        if got.wagons_count != exp["wagons_count"]:
            hint = " (объём задан в тоннах — пересчёт в вагоны это додумывание, нужен null)" \
                if exp["wagons_count"] is None else ""
            bad.append(
                f"заявка {i}: {_fmt(got.wagons_count)} вагонов, "
                f"ожидалось {_fmt(exp['wagons_count'])}{hint}"
            )
    if bad:
        return False, "; ".join(bad[:4]) + (f" (и ещё {len(bad) - 4})" if len(bad) > 4 else "")
    return True, "количество вагонов совпало с эталоном, включая null в заявке 4"


def _check_breaking_no_year():
    """Заявка 1: «в начале мая» — год не указан, период нечёткий."""
    _need_results()
    exp = EXPECTED[1]
    o = RESULTS.get(1)
    if o is None:
        return False, "заявка 1 не разобрана"
    problems = []
    if o.date_from is None or o.date_to is None:
        problems.append("период не заполнен: нужны обе границы date_from и date_to")
    else:
        if o.date_from.year != TODAY.year or o.date_to.year != TODAY.year:
            problems.append(
                f"год {o.date_from.year}/{o.date_to.year} вместо {TODAY.year} — модель взяла год "
                f"из своих знаний, а не из переданного «сегодня» {TODAY.isoformat()}"
            )
        if o.date_from == o.date_to:
            problems.append("«начало мая» схлопнуто в одну дату — нужен диапазон, период не додумываем")
        if not _date_ok(o.date_from, exp["date_from"], exp["date_from_tolerance_days"]):
            problems.append(f"date_from = {o.date_from}, ожидалось около {exp['date_from']}")
        if not _date_ok(o.date_to, exp["date_to"], exp["date_to_tolerance_days"]):
            problems.append(f"date_to = {o.date_to}, ожидалось около {exp['date_to']}")
    if not o.is_ambiguous:
        problems.append("не поднят флаг is_ambiguous — год клиентом не назван")
    elif not (o.ambiguity_reason or "").strip():
        problems.append("флаг поднят, но ambiguity_reason пуста")
    elif not any(k in (o.ambiguity_reason or "").lower() for k in exp["ambiguity_keywords"]):
        problems.append(f"ambiguity_reason не объясняет неоднозначность периода: {o.ambiguity_reason!r}")
    if problems:
        return False, "; ".join(problems)
    return True, (f"год {TODAY.year}, период {o.date_from} … {o.date_to}, флаг поднят "
                  f"с причиной «{(o.ambiguity_reason or '')[:60]}»")


def _check_breaking_range():
    """Заявка 6: период задан диапазоном с точными границами."""
    _need_results()
    exp = EXPECTED[6]
    o = RESULTS.get(6)
    if o is None:
        return False, "заявка 6 не разобрана"
    problems = []
    if o.date_from is None or o.date_to is None:
        problems.append("период не заполнен: нужны обе границы")
    else:
        if o.date_from == o.date_to:
            problems.append(
                "диапазон схлопнут в одну дату — заявка на погрузку с 12 по 18 мая, "
                "а не на один день"
            )
        if not _date_ok(o.date_from, exp["date_from"], exp["date_from_tolerance_days"]):
            problems.append(f"date_from = {o.date_from}, ожидалось {exp['date_from']}")
        if not _date_ok(o.date_to, exp["date_to"], exp["date_to_tolerance_days"]):
            problems.append(f"date_to = {o.date_to}, ожидалось {exp['date_to']}")
    if problems:
        return False, "; ".join(problems)
    return True, f"диапазон сохранён: {o.date_from} … {o.date_to}"


CHECKS = [
    ("Схема TransportOrder: обязательные поля есть, лишние запрещены", _check_schema),
    ("TODO 1: наивный промпт собран и выполнен", _check_todo1),
    ("TODO 2: инженерный промпт с датой, справочниками и правилом", _check_todo2),
    ("TODO 4: нормализация станций к справочнику", _check_todo4),
    ("TODO 3: все 10 заявок разобраны и прошли валидацию", _check_all_parsed),
    ("Станции всех заявок совпали с эталоном", _check_stations),
    ("Груз и род вагона совпали с эталоном", _check_cargo_and_wagon),
    ("Количество вагонов совпало с эталоном", _check_counts),
    ("Ломающая заявка 1: год из «сегодня», диапазон, флаг неоднозначности", _check_breaking_no_year),
    ("Ломающая заявка 6: диапазон 12–18 мая не схлопнут", _check_breaking_range),
]


def check(verbose: bool = True) -> int:
    """Проверяет работу и печатает построчный отчёт. Критерий сдачи — 10/10."""
    passed = 0
    lines = []
    for n, (title, fn) in enumerate(CHECKS, 1):
        try:
            ok, message = fn()
        except TodoNotDone as e:
            ok, message = False, f"{e}. Допишите помеченную ячейку и выполните её заново."
        except AssertionError as e:
            ok, message = False, str(e)
        except Exception as e:  # noqa: BLE001
            ok, message = False, f"проверка упала: {type(e).__name__}: {e}"
        passed += bool(ok)
        lines.append(f"{'✅' if ok else '❌'} {n:>2}. {title}\n      {message}")

    if verbose:
        print("═" * 78)
        print("ПРОВЕРКА ПРАКТИКИ 1 — «Первый вызов LLM и структурированный вывод»")
        print("═" * 78)
        print("\n".join(lines))
        print("─" * 78)
        print(f"ИТОГ: {passed}/{len(CHECKS)}")
        if passed == len(CHECKS):
            print("Работа сдана. Сохраните ноутбук и отправьте PR по инструкции из README.")
        else:
            print("Читайте текст под каждым ❌ — там написано, что именно поправить.")
            print("После правки заново выполните изменённую ячейку, ячейку прогона и check().")
        print("═" * 78)
    return passed


check()

══════════════════════════════════════════════════════════════════════════════
ПРОВЕРКА ПРАКТИКИ 1 — «Первый вызов LLM и структурированный вывод»
══════════════════════════════════════════════════════════════════════════════
✅  1. Схема TransportOrder: обязательные поля есть, лишние запрещены
      поля на месте, лишние запрещены (extra='forbid'), объявлено 10
✅  2. TODO 1: наивный промпт собран и выполнен
      промпт собран (278 символов) и выполнен; модель вернула 'дата' = 'начало мая' — года нет и здесь: период остался словами, одно поле «дата» диапазон не вмещает
✅  3. TODO 2: инженерный промпт с датой, справочниками и правилом
      промпт содержит дату, три справочника и правило, длина 2165 символов
✅  4. TODO 4: нормализация станций к справочнику
      9 случаев нормализации разобраны верно
✅  5. TODO 3: все 10 заявок разобраны и прошли валидацию
      все 10 заявок разобраны и прошли валидацию Pydantic
✅  6. Станции всех заявок совпали с эталоном
      все 20 станций нормализо

10

Шкалу выбрал обычную от 1 до 0, начиная с 1.0 и далее ниже, снимаем немного за каждую причину неопределённости, например, если период нечёткий или если модель не смогла заполнить какое-либо поле. У заявок 1, 4 и 5 confidence получился    0.75, 0.90 и 0.85. У первой снял больше всего, потому что там прямо в тексте «в начале мая» без конкретной даты, а у 4 и 5 просто не хватало одного поля (вагон или количество), поэтому шкала уверенность и не сильно упала. У заявок 2, 3 и 7, где всё в заявке было сказано прямо, confidence получился 0.95–1.00, из чего видим, что между плохими и хорошими заявками разрыв заметный и не пересекается. Я думаю адекватно было бы поставить порог на 0.8, если ниже, то агент не должен автоматически подтверждать заявку и идти дальше, а должен либо переспросить клиента, либо передать клиента на ручного оператора. Лучше сразу уточнить, а не нести потом убытки из-за неправильного принятия заявки агентом.